In [1]:
import time
notebook_start = time.perf_counter()

import os, json, pandas as pd, numpy as np, joblib, shap
import kditransform
from thermoift import print_model_metrics
from thermoift.rng_utils import get_rng
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.inspection import permutation_importance
from tabpfn_extensions.interpretability.shapiq import get_tabpfn_imputation_explainer
from tabpfn_extensions.interpretability.shap import shapiq_to_shap_explanation

In [2]:
PLOT_FOLDER = "TabPFN_P_dew_OUTPUTS"
target      = "P_dew"
SEED        = 6702315
TEST_ROWS   = None
N_TRIALS   = 100

In [3]:
# Parameters
PLOT_FOLDER = "/gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMDew"
TEST_ROWS = None
SEED = 6702315
N_TRIALS = 100


In [4]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="TabPFN fit/predict failed at leaf",
    category=UserWarning,
    module="tabpfn_extensions",
)
os.environ["TABPFN_ALLOW_CPU_LARGE_DATASET"] = "1"

try:
    import tabpfn
    from tabpfn import TabPFNRegressor
    from tabpfn.constants import ModelVersion
    from tabpfn_extensions.hpo import TunedTabPFNRegressor

    print(f"TabPFN version: {tabpfn.__version__}")
    print("Selected model version:", ModelVersion.V3)

except ImportError as exc:
    raise ImportError("tabpfn is not installed in this Python environment.") from exc

n_cpus = int(os.environ.get("SLURM_CPUS_PER_TASK", 4))
os.environ["OMP_NUM_THREADS"]      = str(n_cpus)
os.environ["MKL_NUM_THREADS"]      = str(n_cpus)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cpus)
os.environ["NUMEXPR_NUM_THREADS"]  = str(n_cpus)
try:
    import torch
    torch.set_num_threads(n_cpus)
except ImportError:
    pass
print(f"Thread limit set to {n_cpus} (SLURM_CPUS_PER_TASK)")

TabPFN version: 8.0.3
Selected model version: ModelVersion.V3
Thread limit set to 16 (SLURM_CPUS_PER_TASK)


In [5]:
df = pd.read_csv("../../DATASET_A4/interfacial_results_dataset_A4.csv")
print(f"Number of rows: {len(df)}")

if isinstance(TEST_ROWS, str) and TEST_ROWS.strip().lower() in ("", "none", "null"):
    TEST_ROWS = None
if TEST_ROWS is not None:
    TEST_ROWS = int(TEST_ROWS)
    df = df.iloc[:TEST_ROWS].copy()
    print(f"Test mode: using first {TEST_ROWS} rows only")
else:
    print("Full mode: using all rows")

print(f"Total samples: {len(df)}")
print(f"\n{target} statistics:")
print(df[target].describe())

Number of rows: 19361
Full mode: using all rows
Total samples: 19361

P_dew statistics:
count    19361.000000
mean        23.025120
std         18.245683
min          2.623504
25%          6.876419
50%         16.994768
75%         35.466456
max         78.580621
Name: P_dew, dtype: float64


In [6]:
rng       = get_rng(seed=SEED)

z_columns  = [col for col in df.columns if col.startswith("z_")]
Z_non_zero = [col for col in z_columns if (df[col] != 0).any()]
features   = ["temperature", "pressure"] + Z_non_zero

print(f"Selected features: {features}")

X     = df[features]
y     = df[target]
# 70/15/15 split
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_test,  X_val,  y_test,  y_val  = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

print(f"\nTraining samples:   {X_train.shape[0]}")
print(f"Testing samples:    {X_test.shape[0]}")
print(f"Validation samples: {X_val.shape[0]}")

Selected features: ['temperature', 'pressure', 'z_carbon dioxide', 'z_hydrogen', 'z_nitrogen', 'z_argon', 'z_methane', 'z_oxygen', 'z_carbon monoxide', 'z_hydrogen sulfide']

Training samples:   13552
Testing samples:    2904
Validation samples: 2905


In [7]:
# Train TabPFN model with HPO (100 Bayesian-opt trials)
from tabpfn_extensions.hpo.search_space import get_param_grid_hyperopt
from tabpfn.constants import ModelVersion

_search_space = get_param_grid_hyperopt("regression", model_version=ModelVersion.V2_5)
_search_space["ignore_pretraining_limits"] = True

tuned_model = TunedTabPFNRegressor(
    n_trials=N_TRIALS,
    metric="rmse",
    n_validation_size=0.2,
    shuffle_data=True,
    search_algorithm_type="tpe",
    device="auto",
    random_state=SEED,
    verbose=False,
    search_space=_search_space,
)
tuned_model.fit(X_train, y_train)

y_train_pred = tuned_model.predict(X_train)
y_test_pred  = tuned_model.predict(X_test)
y_val_pred   = tuned_model.predict(X_val)

metrics = print_model_metrics(y_train, y_train_pred, y_test, y_test_pred, target, y_val=y_val, y_val_pred=y_val_pred)

/scratch-local/draju.22814822/ipykernel_3889070/2474745146.py:8: DeprecationWarning: TunedTabPFNRegressor is deprecated and will be removed in a future release of tabpfn-extensions.
  tuned_model = TunedTabPFNRegressor(


Model Performance for P_dew

Training Set:
  R²:   0.999999
  RMSE: 0.017026 bar
  MAE:  0.011687 bar

Test Set:
  R²:   0.999999
  RMSE: 0.018122 bar
  MAE:  0.012283 bar

Validation Set:
  R²:   0.999999
  RMSE: 0.016810 bar
  MAE:  0.011767 bar


In [8]:
from hyperopt import space_eval

# Decode hyperopt indices → actual parameter values
decoded_params = space_eval(_search_space, tuned_model.best_params_)

# Reconstruct inference_config and model_params exactly as the HPO objective does
inference_config = {
    k.split("/")[-1]: v
    for k, v in decoded_params.items()
    if k.startswith("inference_config/")
}
model_params = {
    k: (v.item() if hasattr(v, "item") else v)
    for k, v in decoded_params.items()
    if not k.startswith("inference_config/")
}
model_params.pop("model_type", None)
model_params["inference_config"] = inference_config
model_params["ignore_pretraining_limits"] = True

print("=" * 60)
print(f"Best CV {tuned_model.metric.value.upper()}: {tuned_model.best_score_:.6f}")
print("Best hyperparameter configuration (decoded):")
for k, v in model_params.items():
    print(f"  {k}: {v}")
print("=" * 60)

trials_records = []
for t in tuned_model.trials_.trials:
    if t["result"].get("status") != "ok":
        continue
    row = {"trial_id": t["tid"], "loss": t["result"]["loss"]}
    for param, vals in t["misc"]["vals"].items():
        row[param] = vals[0] if len(vals) else None
    trials_records.append(row)

trials_df = pd.DataFrame(trials_records).sort_values("loss").reset_index(drop=True)
print("\nTop 10 trials:")
print(trials_df.head(10))

os.makedirs(PLOT_FOLDER, exist_ok=True)
trials_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_hpo_trials.csv"), index=False)

# Save joblib: directly usable as TabPFNRegressor(**joblib.load(...))
joblib.dump(model_params, os.path.join(PLOT_FOLDER, f"TabPFN_{target}_decoded_best_config.joblib"))

# Save JSON: human-readable (non-serializable objects fall back to str)
def _to_json(v):
    if hasattr(v, "item"):
        return v.item()
    if isinstance(v, dict):
        return {kk: _to_json(vv) for kk, vv in v.items()}
    if isinstance(v, (list, tuple)):
        return [_to_json(x) for x in v]
    return v

with open(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_best_config.json"), "w") as f:
    json.dump({
        "best_score": float(tuned_model.best_score_),
        "metric": tuned_model.metric.value,
        "encoded_best_params": {
            k: (v.item() if hasattr(v, "item") else v)
            for k, v in tuned_model.best_params_.items()
        },
        "model_params": _to_json(model_params),
    }, f, indent=2, default=str)

print(f"\nSaved to {PLOT_FOLDER}/")
print(f"  TabPFN_{target}_decoded_best_config.joblib  ← use with TabPFNRegressor(**joblib.load(...))")
print(f"  TabPFN_{target}_best_config.json            ← human-readable")

Best CV RMSE: -0.017701
Best hyperparameter configuration (decoded):
  average_before_softmax: False
  ignore_pretraining_limits: True
  model_path: /home/draju/.cache/tabpfn/tabpfn-v2.5-regressor-v2.5_small-samples.ckpt
  n_estimators: 4
  softmax_temperature: 0.8
  inference_config: {'FINGERPRINT_FEATURE': False, 'MIN_UNIQUE_FOR_NUMERICAL_FEATURES': 30, 'OUTLIER_REMOVAL_STD': None, 'POLYNOMIAL_FEATURES': 'no', 'PREPROCESS_TRANSFORMS': ({'append_original': False, 'categorical_name': 'ordinal_very_common_categories_shuffled', 'global_transformer_name': 'svd', 'name': 'kdi_uni'},), 'REGRESSION_Y_PREPROCESS_TRANSFORMS': (None,)}

Top 10 trials:
   trial_id      loss  FINGERPRINT_FEATURE  MIN_UNIQUE_FOR_NUMERICAL_FEATURES  \
0        27  0.017701                    1                                  3   
1        78  0.019274                    1                                  3   
2        62  0.020237                    1                                  3   
3        69  0.020647    

In [9]:
results_df = pd.DataFrame({
    "idx":       np.concatenate([y_train.index, y_test.index, y_val.index]),
    "actual":    np.concatenate([y_train.values, y_test.values, y_val.values]),
    "predicted": np.concatenate([y_train_pred,  y_test_pred,  y_val_pred]),
    "split":     ["train"]*len(y_train) + ["test"]*len(y_test) + ["val"]*len(y_val),
})

os.makedirs(PLOT_FOLDER, exist_ok=True)
results_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_predictions.csv"), index=False)
print(f"Predictions saved: {len(results_df)} rows")

best_tabpfn = tuned_model.best_model_
model_path  = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_model.joblib")
joblib.dump(best_tabpfn, model_path)
print(f"Model saved to: {model_path}")

tuner_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_tuner.joblib")
joblib.dump(tuned_model, tuner_path)
print(f"Tuner saved to: {tuner_path}")

Predictions saved: 19361 rows
Model saved to: /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMDew/TabPFN_P_dew_model.joblib


Tuner saved to: /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMDew/TabPFN_P_dew_tuner.joblib


In [10]:
best_tabpfn = tuned_model.best_model_
cv_results = cross_validate(
    best_tabpfn, X, y, cv=5,
    scoring={
        "r2":   "r2",
        "rmse": "neg_root_mean_squared_error",
        "mae":  "neg_mean_absolute_error",
    },
    n_jobs=n_cpus,
)

cv_r2_scores   = cv_results["test_r2"]
cv_rmse_scores = -cv_results["test_rmse"]
cv_mae_scores  = -cv_results["test_mae"]

print(f"Cross-Validation R² Scores:   {cv_r2_scores}")
print(f"Mean CV R²:   {cv_r2_scores.mean():.6f} (+/- {cv_r2_scores.std() * 2:.6f})")
print(f"\nCross-Validation RMSE Scores: {cv_rmse_scores}")
print(f"Mean CV RMSE: {cv_rmse_scores.mean():.6f} (+/- {cv_rmse_scores.std() * 2:.6f})")
print(f"\nCross-Validation MAE Scores:  {cv_mae_scores}")
print(f"Mean CV MAE:  {cv_mae_scores.mean():.6f} (+/- {cv_mae_scores.std() * 2:.6f})")

Cross-Validation R² Scores:   [0.99975646 0.99964086 0.98277727 0.99890489 0.99963433]
Mean CV R²:   0.996143 (+/- 0.013379)

Cross-Validation RMSE Scores: [0.27851487 0.34523836 2.40261094 0.61086129 0.35148831]
Mean CV RMSE: 0.797743 (+/- 1.620871)

Cross-Validation MAE Scores:  [0.1636138  0.18634928 0.22300927 0.23972552 0.20334058]
Mean CV MAE:  0.203208 (+/- 0.053498)


In [11]:
best_tabpfn = tuned_model.best_model_
perm = permutation_importance(
    best_tabpfn,
    X_test,
    y_test,
    n_repeats=10,
    random_state=SEED,
    scoring="neg_root_mean_squared_error",
    n_jobs=n_cpus,
)

perm_df = (
    pd.DataFrame({
        "feature":         features,
        "importance_mean": perm.importances_mean,
        "importance_std":  perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)
print("Permutation importance (drop in score when feature is shuffled):")
print(perm_df)

os.makedirs(PLOT_FOLDER, exist_ok=True)
perm_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_permutation_importance.csv"), index=False)

X_explain = X_test

_n_features = len(features)
_budget = min(2 ** _n_features, 512)
_explainer = get_tabpfn_imputation_explainer(
    best_tabpfn, X_train, index="SV", max_order=1
)
shap_values = shapiq_to_shap_explanation(
    _explainer, X_explain, budget=_budget, feature_names=features
)

shap_arr = shap_values.values if hasattr(shap_values, "values") else np.asarray(shap_values)
if shap_arr.ndim == 3:
    shap_arr = shap_arr[:, :, 0]
mean_abs = np.abs(shap_arr).mean(axis=0)

shap_rank_df = (
    pd.DataFrame({"feature": features, "mean_abs_shap": mean_abs})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_rank_df.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_importance.csv"), index=False)
print("Mean |SHAP| ranking:")
print(shap_rank_df)

# Save artifacts for the plotting notebook
joblib.dump(shap_values, os.path.join(PLOT_FOLDER, f"TabPFN_{target}_shap_values.joblib"))
X_explain.to_csv(os.path.join(PLOT_FOLDER, f"TabPFN_{target}_X_explain.csv"), index=True)
print(f"SHAP values + X_explain saved to {PLOT_FOLDER}/")

metrics["permutation_importance"] = perm_df.to_dict(orient="records")
metrics["shap_importance"]        = shap_rank_df.to_dict(orient="records")

/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


/home/draju/A6/.TPFN/lib/python3.13/site-packages/tabpfn/inference.py:1334: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  X_train = torch.as_tensor(X_train, dtype=dtype, device=device)


/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


...

/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(
/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


/home/draju/A6/.TPFN/lib/python3.13/site-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but TabPFNRegressor was fitted without feature names
  warnings.warn(


Permutation importance (drop in score when feature is shuffled):
              feature  importance_mean  importance_std
0         temperature        25.971682        0.238662
1             z_argon         0.593867        0.015457
2            pressure         0.503172        0.024027
3           z_methane         0.464128        0.016293
4          z_nitrogen         0.441963        0.012397
5          z_hydrogen         0.413028        0.011321
6   z_carbon monoxide         0.339615        0.031932
7            z_oxygen         0.310210        0.008400
8  z_hydrogen sulfide         0.302000        0.022159
9    z_carbon dioxide         0.238124        0.014425


/home/draju/A6/.TPFN/lib/python3.13/site-packages/tabpfn_common_utils/telemetry/core/decorators.py:218: UserWarning: TabPFN model has fit_mode='fit_preprocessors', not 'fit_with_cache'. Imputation-based SHAP will be substantially slower than necessary. Construct the model with TabPFNClassifier or TabPFNRegressor (fit_mode='fit_with_cache', ...) (set BEFORE calling .fit) to enable the KV cache, then set model.executor_.keep_cache_on_device = True after .fit().
  return fn(*args, **kwargs)


Mean |SHAP| ranking:
              feature  mean_abs_shap
0         temperature      21.087810
1             z_argon       0.120696
2          z_nitrogen       0.107849
3           z_methane       0.088305
4          z_hydrogen       0.087179
5   z_carbon monoxide       0.075702
6            z_oxygen       0.066907
7    z_carbon dioxide       0.056638
8            pressure       0.053405
9  z_hydrogen sulfide       0.043594
SHAP values + X_explain saved to /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMDew/


In [12]:
metrics["cv_r2_scores"]    = cv_r2_scores.tolist()
metrics["cv_r2_mean"]      = float(cv_r2_scores.mean())
metrics["cv_r2_std"]       = float(cv_r2_scores.std())
metrics["cv_rmse_scores"]  = cv_rmse_scores.tolist()
metrics["cv_rmse_mean"]    = float(cv_rmse_scores.mean())
metrics["cv_rmse_std"]     = float(cv_rmse_scores.std())
metrics["cv_mae_scores"]   = cv_mae_scores.tolist()
metrics["cv_mae_mean"]     = float(cv_mae_scores.mean())
metrics["cv_mae_std"]      = float(cv_mae_scores.std())
metrics["model"]           = "TabPFN"
metrics["features"]        = features
metrics["target"]          = target
metrics["seed"]            = SEED
metrics["best_hyperparameters"] = {
    k: (v.item() if hasattr(v, "item") else v)
    for k, v in tuned_model.best_params_.items()
}
metrics["best_hpo_score"] = float(tuned_model.best_score_)
metrics["hpo_metric"]     = tuned_model.metric.value
metrics["hpo_n_trials"]   = tuned_model.n_trials

os.makedirs(PLOT_FOLDER, exist_ok=True)
metrics_path = os.path.join(PLOT_FOLDER, f"TabPFN_{target}_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2, default=str)
print(f"\nMetrics saved to: {metrics_path}")


Metrics saved to: /gpfs/home6/draju/A6/TabPFN_V3/with_HPO/SLURMDew/TabPFN_P_dew_metrics.json


In [13]:
notebook_end = time.perf_counter()
elapsed_minutes = (notebook_end - notebook_start) / 60
print(f"Total notebook runtime: {elapsed_minutes:.2f} minutes")

Total notebook runtime: 18.16 minutes
